In [18]:
!pip install yfinance

In [19]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import yfinance as yf

In [21]:
%pip install --quiet numpy matplotlib scipy pandas scikit-learn yfinance

import os
import sys
import importlib
from pathlib import Path

repo_path = "/content/Quant-Finance-Summer-2026"

if not os.path.isdir(os.path.join(repo_path, ".git")):
    !git clone https://github.com/munawarali93/Quant-Finance-Summer-2026.git "$repo_path"
else:
    !git -C "$repo_path" pull

if repo_path in sys.path:
    sys.path.remove(repo_path)

sys.path.insert(0, repo_path)

for module_name in list(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        del sys.modules[module_name]

importlib.invalidate_caches()


estimator_path = Path(repo_path) / "src" / "estimator.py"
print("Importing from:", estimator_path)

from src.estimator import *

print("Successfully imported estimator functions.")

Already up to date.
Importing from: /content/Quant-Finance-Summer-2026/src/estimator.py
Successfully imported estimator functions.


##Experiment 1: Verifiying the estimator on fractional Brownian motion
##Simulates many fBM paths for several true H values.
##For each H, it estimates H_hat over n_paths independent paths.

In [22]:
def monte_carlo_fbm_many_H(
    H_values=[0.1, 0.3, 0.5, 0.8],
    N=1000,
    T=1.0,
    n_paths=100,
    K=None,
    seed=123
):
    np.random.seed(seed)

    summaries = []

    for H_true in H_values:
        estimates = []

        for m in range(n_paths):
            B = davies_harte(N=N, T=T, H=H_true)
            B = B[:-1].real.astype(float)
            H_hat = estimate_H(B, K=K, T=T)
            estimates.append(H_hat)
        estimates = np.array(estimates, dtype=float)
        estimates = estimates[np.isfinite(estimates)]

        summary = {
            "H_true": H_true,
            "N": N,
            "T": T,
            "K": int(K) if K is not None else int(np.floor(np.sqrt(len(B) - 1))),
            "n_paths": len(estimates),
            "min": np.min(estimates),
            "mean": np.mean(estimates),
            "std": np.std(estimates, ddof=1),
            "max": np.max(estimates),
            "bias": np.mean(estimates) - H_true,
            "rmse": np.sqrt(np.mean((estimates - H_true) ** 2)),
            "estimates": estimates
        }

        summaries.append(summary)

    return summaries



#converting to latex table code
def summaries_to_latex_table(summaries, digits=4):
    rows = ""
    for s in summaries:
        rows += (
            f'{s["H_true"]:.1f} & '
            f'{s["min"]:.{digits}f} & '
            f'{s["mean"]:.{digits}f} & '
            f'{s["std"]:.{digits}f} & '
            f'{s["max"]:.{digits}f} \\\\\n'
        )

    N = summaries[0]["N"]
    K = summaries[0]["K"]
    n_paths = summaries[0]["n_paths"]

    latex = rf"""
\begin{{table}}[H]
\centering
\begin{{tabular}}{{ccccc}}
\hline
$H$ & Min. & Mean & Std. & Max. \\
\hline
{rows}\hline
\end{{tabular}}
\caption{{Summary statistics for the estimated roughness index $\widehat{{H}}_{{L,K}}$ for fractional Brownian motion. The simulations use $N={N}$, $K={K}$, and {n_paths} independent paths for each value of $H$.}}
\end{{table}}
"""
    return latex

H_values = [0.1, 0.3, 0.5, 0.8]

N = 10000
T = 1.0
n_paths = 50

K = int(np.floor(np.sqrt(N - 1)))

summaries = monte_carlo_fbm_many_H(
    H_values=H_values,
    N=N,
    T=T,
    n_paths=n_paths,
    K=K,
    seed=123
)

for s in summaries:
    print(
        f'H={s["H_true"]:.1f}, '
        f'Min={s["min"]:.4f}, '
        f'Mean={s["mean"]:.4f}, '
        f'Std={s["std"]:.4f}, '
        f'Max={s["max"]:.4f}'
    )

print("\nLaTeX code:")
print(summaries_to_latex_table(summaries, digits=4))

H=0.1, Min=0.0833, Mean=0.1060, Std=0.0201, Max=0.1624
H=0.3, Min=0.2460, Mean=0.2991, Std=0.0197, Max=0.3460
H=0.5, Min=0.4693, Mean=0.5036, Std=0.0117, Max=0.5273
H=0.8, Min=0.7379, Mean=0.7848, Std=0.0206, Max=0.8447

LaTeX code:

\begin{table}[H]
\centering
\begin{tabular}{ccccc}
\hline
$H$ & Min. & Mean & Std. & Max. \\
\hline
0.1 & 0.0833 & 0.1060 & 0.0201 & 0.1624 \\
0.3 & 0.2460 & 0.2991 & 0.0197 & 0.3460 \\
0.5 & 0.4693 & 0.5036 & 0.0117 & 0.5273 \\
0.8 & 0.7379 & 0.7848 & 0.0206 & 0.8447 \\
\hline
\end{tabular}
\caption{Summary statistics for the estimated roughness index $\widehat{H}_{L,K}$ for fractional Brownian motion. The simulations use $N=10000$, $K=99$, and 50 independent paths for each value of $H$.}
\end{table}

